# P1 · Salarios de Datos e IA vs. resto de IT, comparados con el piso del convenio AGC

**Equipo 3 – PP1 Economía del Conocimiento – 2C 2026**

**Pregunta 1:** ¿Cuánto cobran los puestos vinculados a datos e inteligencia artificial en comparación con el resto de los puestos IT?

Este notebook compara la mediana salarial de **Datos e IA** y **Resto IT** (Sysarmy 2026.1) con el **piso del convenio CCT 1696/23 de la AGC** (enero 2026, estimado a partir del Anexo 3T 2026; ver notebook de validación de David).

**Cómo usarlo en Google Colab:**
1. Subí este notebook (*Archivo → Subir cuaderno*).
2. En el panel de la izquierda (ícono de carpeta 📁 → ⬆), subí `sysarmy_limpio.csv`.
3. *Entorno de ejecución → Ejecutar todas*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

df = pd.read_csv("../data/processed/sysarmy_limpio.csv")
print(f"Respuestas cargadas: {len(df):,}")

## 1. Filtro: quienes pueden estar bajo convenio

Para comparar con un piso de convenio en pesos se conservan solo:
- **Full-Time** (el piso es para jornada completa),
- sueldo **en pesos** (`sueldo_dolarizado = 0`),
- **Staff** o **Tercerizado** (relación de dependencia; se excluyen contractors y freelance).

In [ ]:
CONTRATOS = ["Staff (planta permanente)", "Tercerizado (trabajo a través de consultora o agencia)"]
conv = df[(df["dedicacion"] == "Full-Time") & (df["sueldo_dolarizado"] == 0) & (df["tipo_contrato"].isin(CONTRATOS))]
print(f"Respuestas comparables con el convenio: {len(conv):,}")
conv.groupby(["seniority", "grupo_puesto"]).size().unstack()

## 2. Piso del convenio AGC (enero 2026)

Básico de **Programador/a** y **Analista de Datos** (en el convenio tienen el mismo valor) y básico + 20 % no remunerativo.
Valores **estimados**: reconstruidos desde el Anexo 3T 2026 deshaciendo los aumentos publicados por la AGC (ver `validacion_salarios_sysarmy.ipynb`).

In [ ]:
grilla = pd.DataFrame({
    "seniority": ["Junior", "Semi-Senior", "Senior"],
    "piso_basico": [1424494, 1638169, 1883894],
})
grilla["piso_con_no_rem"] = (grilla["piso_basico"] * 1.20).round()
grilla

## 3. Medianas por seniority y comparación con el piso

In [ ]:
NIVELES = ["Junior", "Semi-Senior", "Senior"]
tabla = (conv.pivot_table(index="seniority", columns="grupo_puesto", values="salario_bruto", aggfunc="median")
             .reindex(NIVELES)
             .merge(grilla.set_index("seniority"), left_index=True, right_index=True))

for g in ["Datos e IA", "Resto IT"]:
    tabla[f"{g} vs piso (+20%)"] = (tabla[g] / tabla["piso_con_no_rem"] - 1) * 100
    tabla[f"% {g} bajo el piso"] = [
        (conv[(conv.grupo_puesto == g) & (conv.seniority == s)].salario_bruto < tabla.loc[s, "piso_con_no_rem"]).mean() * 100
        for s in NIVELES]

tabla.round(1)

## 4. Gráfico

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(NIVELES)); w = 0.27
series = [("Datos e IA", "Datos e IA", "#1f77b4"),
          ("Resto IT", "Resto IT", "#A0A0A0"),
          ("piso_con_no_rem", "Piso AGC (básico + 20%)", "#ff7f0e")]
for i, (col, etiqueta, color) in enumerate(series):
    vals = tabla[col].values
    barras = ax.bar([p + (i - 1) * w for p in x], vals, w, label=etiqueta, color=color)
    ax.bar_label(barras, labels=[f"${v/1e6:.2f} M" for v in vals], padding=3, fontsize=9)

ax.set_xticks(list(x)); ax.set_xticklabels(NIVELES)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v/1e6:.1f} M"))
ax.set_ylabel("Salario bruto mensual (ARS)")
ax.set_title("Datos e IA supera el piso del convenio por más margen que el resto de IT, sobre todo en Junior", fontsize=12)
ax.legend(loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, 0.01, "Sysarmy 2026.1 · Full-Time, sueldo en pesos, staff/tercerizado (n = %d). Piso AGC CCT 1696/23, enero 2026 (estimado)." % len(conv),
         fontsize=8, color="gray")
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig("p1_datos_ia_vs_convenio.png", dpi=200)
plt.show()

## 5. Hallazgo e insight

**Hallazgo:** en **Junior**, la mediana de **Datos e IA** está ~24 % por encima del piso del convenio (con el 20 % no remunerativo), mientras que **Resto IT** está apenas ~2 % arriba. Cerca de la mitad de los Junior de Resto IT declara menos que el piso, frente a ~1 de cada 3 en Datos e IA. En Semi-Senior y Senior ambos grupos superan ampliamente el piso.

**Insight:** el convenio fija **el mismo piso** para Analista de Datos que para Programador/a: la ventaja salarial de Datos e IA no proviene del convenio sino del mercado, y es más marcada en los niveles de entrada.

**Limitaciones:** encuesta voluntaria y autodeclarada (51 % de respuestas de CABA); piso de enero 2026 estimado; se desconoce si `salario_bruto` incluye el 20 % no remunerativo; pisos sin adicionales (antigüedad, título, idioma).